# 05 · Exit sensitivity (research IS only)

Other test — **not** a STAR change. Symmetric `exit_z` grid and separable stop
mechanisms (HL timeout, ATR path stop, percentage pair max-loss), alone and in combination.

STAR / ledger / hypothesis notebooks unchanged. Self-contained config overrides only.

## 0. Imports & Config

In [ ]:
from __future__ import annotations

import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from IPython.display import display

from backtest.s2_coint.diagnosis import enrich_trades, extreme_trades
from backtest.s2_coint.report import load_star_stack, require_star
from backtest.s2_coint.research import (
    DEFAULT_STAR_STACK,
    config_from_stack,
    frozen_pairs_for_universe,
    is_end_for_stack,
    load_s1_weekly,
    load_star_panels,
    load_universe_panels,
    lookbacks_for_bar,
    overlay_kalman_hedge,
    overlay_ols_hedge,
    split_is_oos,
)
from backtest.s2_coint.tearsheet import cvar, fit_mean_abs_score
from strategies.s2_coint.engine import simulate_book
from strategies.s2_coint.metrics import corr_to_s1, metrics_from_returns_inference
from strategies.s2_coint.sizing import pair_scale_from_score

warnings.filterwarnings("ignore", category=FutureWarning)

N_EXTREME = 7
STAR_PATH = DEFAULT_STAR_STACK
stack = load_star_stack(STAR_PATH)
require_star("UNIVERSE_STAR", stack.get("UNIVERSE_STAR"))
require_star("EXIT_STAR", stack.get("EXIT_STAR"))
UNIVERSE = str(stack["UNIVERSE_STAR"])
PAIRS = list(stack.get("PAIRS_STAR") or frozen_pairs_for_universe(UNIVERSE, "1d", root=ROOT))
print("UNIVERSE", UNIVERSE)
print("PAIRS", PAIRS)
print("EXIT_STAR", stack.get("EXIT_STAR"))
print("BREAK_STAR", stack.get("BREAK_STAR"))
print("SIZE_STAR", stack.get("SIZE_STAR"))
print("TREND_STAR", stack.get("TREND_STAR"))
print("VOL_STAR", stack.get("VOL_STAR"))
print("NOTE: STAR stack is read-only in this notebook (other_tests).")

In [ ]:
def _cagr(returns: pd.Series, periods_per_year: float = 252.0) -> float:
    r = pd.to_numeric(returns, errors="coerce").fillna(0.0).astype(float)
    if r.empty:
        return float("nan")
    total = float((1.0 + r).prod())
    years = len(r) / float(periods_per_year)
    if years <= 0 or total <= 0:
        return float("nan")
    return float(total ** (1.0 / years) - 1.0)


def arm_metrics(returns: pd.Series, s1: pd.Series | None = None) -> dict:
    r = pd.to_numeric(returns, errors="coerce").fillna(0.0).astype(float)
    r.index = pd.to_datetime(r.index)
    m = metrics_from_returns_inference(r, periods_per_year=252.0)
    cagr = _cagr(r)
    mdd = float(m.get("max_drawdown", float("nan")))
    calmar = float(cagr / abs(mdd)) if np.isfinite(cagr) and np.isfinite(mdd) and mdd != 0 else float("nan")
    return {
        "ann_sharpe": m.get("ann_sharpe", float("nan")),
        "max_drawdown": mdd,
        "calmar": calmar,
        "cagr": cagr,
        "skew": m.get("skew", float("nan")),
        "excess_kurtosis": m.get("excess_kurtosis", float("nan")),
        "cvar_5pct": cvar(r, alpha=0.05),
        "corr_to_s1": corr_to_s1(r, s1 if s1 is not None else s1_weekly),
        "n_days": m.get("n_days", 0),
    }


def collect_trades(book, panel: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for pid, res in book.pair_results.items():
        if res.trades is None or res.trades.empty:
            continue
        t = res.trades.copy()
        t["pair_id"] = str(pid)
        frames.append(enrich_trades(t, panel, res.returns))
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def show_extreme(trades: pd.DataFrame, n: int = N_EXTREME, title: str = "") -> None:
    best, worst = extreme_trades(trades, n=n)
    cols = [
        "pair_id", "side_label", "entry_date", "exit_date", "hold_bars",
        "exit_reason", "pnl_pct", "z_entry", "z_exit", "adf_entry", "adf_exit",
    ]
    if title:
        print(title)
    print(f"=== Top {n} trades ===")
    display(best[cols] if not best.empty else best)
    print(f"=== Bottom {n} trades ===")
    display(worst[cols] if not worst.empty else worst)


def metrics_table(rows: dict[str, dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows).T
    order = [
        "ann_sharpe", "max_drawdown", "calmar", "cagr", "skew",
        "excess_kurtosis", "cvar_5pct", "corr_to_s1", "n_days",
    ]
    cols = [c for c in order if c in df.columns] + [c for c in df.columns if c not in order]
    return df[cols]

from dataclasses import replace
from strategies.s2_coint.spread_ohlc import attach_spread_indicators

In [ ]:
bar = str(stack.get("BAR_STAR") or "1d")
lb = lookbacks_for_bar(bar)
# Prefer cached STAR panels when present; else overlay hedge on the research-IS train panel only.
try:
    train_star, _full_star, _manifest = load_star_panels(
        universe=UNIVERSE, bar=bar, pair_ids=PAIRS, root=ROOT
    )
    is_end = is_end_for_stack(stack, train_star)
    is_raw, _oos_unused = split_is_oos(train_star, is_end=is_end)
    del _oos_unused
    panel_src = "cached star train"
except (FileNotFoundError, ValueError) as exc:
    print("star panels unavailable (", type(exc).__name__, ") — using universe train panel")
    train, _full = load_universe_panels(UNIVERSE, bar, PAIRS, root=ROOT)
    is_end = is_end_for_stack(stack, train)
    is_raw, _oos_unused = split_is_oos(train, is_end=is_end)
    del _oos_unused
    panel_src = "universe train"

hedge = str(stack.get("HEDGE_STAR") or "ols")
# Skip re-overlay when star cache already has z / adf columns.
need_overlay = "z" not in is_raw.columns or "adf_pvalue" not in is_raw.columns
if need_overlay:
    if hedge == "kalman":
        is_panel = overlay_kalman_hedge(
            is_raw,
            z_window=lb["z_window"],
            hl_window=lb["hl_window"],
            adf_window=lb["adf_window"],
        )
    else:
        is_panel = overlay_ols_hedge(
            is_raw,
            ols_window=lb["ols_window"],
            z_window=lb["z_window"],
            hl_window=lb["hl_window"],
            adf_window=lb["adf_window"],
        )
else:
    is_panel = is_raw.copy()

is_panel = is_panel.loc[is_panel["pair_id"].astype(str).isin(PAIRS)].copy()
is_panel["date"] = pd.to_datetime(is_panel["date"])

mean_abs = fit_mean_abs_score(is_panel, score_column="z")
s1_weekly = load_s1_weekly(ROOT)
cfg_star = config_from_stack(stack)
print("panel_src", panel_src, "need_overlay", need_overlay)
print("bar", bar, "is_end", is_end)
print("IS rows", len(is_panel), "pairs", is_panel["pair_id"].nunique())
print("frozen IS mean(|z|)", round(mean_abs, 4))
print("cfg", cfg_star)

In [ ]:
panel_base = is_panel.copy()
panel_ohlc = attach_spread_indicators(is_panel, atr_window=int(cfg_star.atr_window))
print("panel_base cols has atr_spread?", "atr_spread" in panel_base.columns)
print("panel_ohlc cols has atr_spread?", "atr_spread" in panel_ohlc.columns)

## 1. Baseline

In [ ]:
results = {}
books = {}
trades_by_arm = {}

def run_arm(name: str, cfg, panel) -> None:
    book = simulate_book(panel, cfg, mean_abs_score=mean_abs)
    books[name] = book
    results[name] = arm_metrics(book.returns)
    trades_by_arm[name] = collect_trades(book, panel)
    print(name, {k: round(v, 4) if isinstance(v, float) else v for k, v in results[name].items()})

run_arm("baseline_mean_only", cfg_star, panel_base)
show_extreme(trades_by_arm["baseline_mean_only"], title="Baseline extremes")

## 2. exit_z Sensitivity (symmetric only)

In [ ]:
exit_z_arms = []
for ez in (0.0, 0.1, 0.2, 0.3):
    name = f"exit_z_{ez}"
    cfg = replace(cfg_star, exit_z=float(ez), exit_mode="mean_only")
    run_arm(name, cfg, panel_base)
    exit_z_arms.append(name)

ez_keys = [k for k in results if k.startswith("exit_z_") or k == "baseline_mean_only"]
display(metrics_table({k: results[k] for k in ez_keys}))

# Distribution plots for each exit_z arm (daily book returns)
fig, axes = plt.subplots(2, 2, figsize=(10, 6.5), sharex=True, sharey=True)
axes = axes.ravel()
for ax, name in zip(axes, exit_z_arms):
    r = books[name].returns.astype(float).dropna()
    ax.hist(r, bins=40, color="#1f4e79", alpha=0.85, density=True)
    ax.axvline(0.0, color="k", lw=0.8)
    ax.set_title(
        f"{name}\nskew={results[name]['skew']:.2f}  "
        f"exkurt={results[name]['excess_kurtosis']:.1f}"
    )
    ax.grid(True, alpha=0.3)
plt.suptitle("exit_z daily return distributions (research IS)", y=1.01)
plt.tight_layout()
plt.show()


## 3. Individual Stop Mechanisms

Isolate one mechanism via `exit_mode=hl3_atr_breaker` + extreme parameters to disable the others.

| Arm | Active |
|-----|--------|
| hl | HL timeout only (`n_half_lives=3`, pct disabled, no ATR cols) |
| atr | ATR stop only (HL disabled, pct disabled, OHLC panel) |
| pct3 / pct5 / pct7 | Pair capital stop only at −3/−5/−7% |

In [ ]:
DISABLE_HL = 1e9
DISABLE_PCT = -1e9

indiv = {
    "hl": dict(n_half_lives=3.0, pair_max_loss=DISABLE_PCT, panel=panel_base),
    "atr": dict(n_half_lives=DISABLE_HL, pair_max_loss=DISABLE_PCT, panel=panel_ohlc),
    "pct3": dict(n_half_lives=DISABLE_HL, pair_max_loss=-0.03, panel=panel_base),
    "pct5": dict(n_half_lives=DISABLE_HL, pair_max_loss=-0.05, panel=panel_base),
    "pct7": dict(n_half_lives=DISABLE_HL, pair_max_loss=-0.07, panel=panel_base),
}
for name, kw in indiv.items():
    panel = kw["panel"]
    cfg = replace(
        cfg_star,
        exit_mode="hl3_atr_breaker",
        exit_z=0.0,
        n_half_lives=float(kw["n_half_lives"]),
        pair_max_loss=float(kw["pair_max_loss"]),
    )
    run_arm(name, cfg, panel)
    # exit reason mix
    t = trades_by_arm[name]
    if not t.empty:
        print(name, "exit_reason counts:", t["exit_reason"].value_counts().to_dict())

display(metrics_table({k: results[k] for k in ["baseline_mean_only", *indiv]}))

## 4. Combination Arms (pct5 as representative)

In [ ]:
combos = {
    "hl+pct5": dict(n_half_lives=3.0, pair_max_loss=-0.05, panel=panel_base),
    "hl+atr": dict(n_half_lives=3.0, pair_max_loss=DISABLE_PCT, panel=panel_ohlc),
    "atr+pct5": dict(n_half_lives=DISABLE_HL, pair_max_loss=-0.05, panel=panel_ohlc),
    "hl+atr+pct5": dict(n_half_lives=3.0, pair_max_loss=-0.05, panel=panel_ohlc),
}
for name, kw in combos.items():
    panel = kw["panel"]
    cfg = replace(
        cfg_star,
        exit_mode="hl3_atr_breaker",
        exit_z=0.0,
        n_half_lives=float(kw["n_half_lives"]),
        pair_max_loss=float(kw["pair_max_loss"]),
    )
    run_arm(name, cfg, panel)
    t = trades_by_arm[name]
    if not t.empty:
        print(name, "exit_reason counts:", t["exit_reason"].value_counts().to_dict())

display(metrics_table({k: results[k] for k in combos}))

## 5. Comparison

In [ ]:
all_tbl = metrics_table(results)
display(all_tbl.sort_values("ann_sharpe", ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, col, title in zip(
    axes,
    ["ann_sharpe", "excess_kurtosis", "cvar_5pct"],
    ["Sharpe", "Excess kurtosis", "CVaR 5%"],
):
    all_tbl[col].plot(kind="bar", ax=ax, color="#1f4e79", title=title)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Extremes for top-Sharpe non-baseline arm and lowest-kurtosis arm
rank_sharpe = all_tbl["ann_sharpe"].sort_values(ascending=False)
rank_kurt = all_tbl["excess_kurtosis"].sort_values()
for label, series in [("best Sharpe", rank_sharpe), ("lowest kurtosis", rank_kurt)]:
    arm = str(series.index[0])
    show_extreme(trades_by_arm[arm], title=f"{label}: {arm}")

## 6. Summary

After running: which exit_z / stop / combo best improves excess kurtosis without wrecking Sharpe / Calmar / corr_to_s1?
Any arm worth promoting later to a formal hypothesis?